# 05 — `__slots__`, `weakref`, garbage collector

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `__slots__` pour économiser de la mémoire et bloquer l'ajout d'attributs ;
- comprendre les interactions de `__slots__` avec l'héritage et `__dict__` ;
- utiliser `weakref` pour éviter les cycles et les caches qui tiennent trop d'objets ;
- diagnostiquer les cycles et les fuites mémoire avec le module `gc` ;
- savoir quand Python utilise le GC et quand il utilise le comptage de références.


## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les type hints modernes (`int | None`, génériques, `Protocol`, `TypeVar`) ;
- les dataclasses (`@dataclass`, `field`, `frozen=True`, `slots=True`) ;
- les décorateurs de fonction et de classe, et les gestionnaires de contexte ;
- les tests avec `pytest` (fixtures, paramétrage, monkeypatch) ;
- le packaging avec `pyproject.toml` et `uv` ;
- le logging (module `logging`, handlers, formatters) ;
- les bases de SQL et `sqlite3`, les expressions régulières.

Notions que nous allons **introduire ou approfondir** ici :

- `__slots__` et ses gains mémoire mesurés ;
- `weakref`, `WeakValueDictionary`, `WeakKeyDictionary` ;
- le GC cyclique de CPython, `gc.collect`, `gc.get_referrers`.


## Plan

1. `__slots__` : motivation et usage
2. Gain mesuré : avec vs sans `__slots__`
3. `__slots__` et héritage
4. `weakref` : références faibles
5. `WeakValueDictionary` et caches
6. Le GC CPython : comptage de références + cycle collector
7. Diagnostiquer une fuite mémoire
8. Synthèse
9. Exercices


---

## 1. `__slots__` — motivation

Par défaut, chaque instance Python porte un `__dict__` : un dictionnaire qui stocke ses attributs. C'est **flexible** (on peut ajouter n'importe quel attribut) mais **coûteux** : un dict vide fait ~100 octets minimum, et l'accès à un attribut passe par une recherche hash.

`__slots__` permet de remplacer ce dict par un tableau à taille fixe, ce qui :

- **économise de la mémoire** (parfois 50 % sur des objets simples),
- **rend l'accès aux attributs plus rapide**,
- **interdit** l'ajout d'attributs non prévus (effet secondaire utile pour la discipline).

In [ ]:
class Normal:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

class Slotte:
    __slots__ = ("x", "y")
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

n = Normal(1, 2)
s = Slotte(1, 2)

# Sur l'objet Normal on peut ajouter un attribut
n.z = 3
print(n.z)

# Sur l'objet Slotte, non
try:
    s.z = 3
except AttributeError as e:
    print("refusé :", e)

---

## 2. Gain mesuré

Mesurons la taille totale d'un lot d'instances :

In [ ]:
import sys

class Normal:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

class Slotte:
    __slots__ = ("x", "y")
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

n = Normal(1.0, 2.0)
s = Slotte(1.0, 2.0)

# getsizeof ne descend pas dans __dict__, on mesure donc à la main
taille_normal = sys.getsizeof(n) + sys.getsizeof(n.__dict__)
taille_slotte = sys.getsizeof(s)
print(f"Normal : {taille_normal} octets")
print(f"Slotte : {taille_slotte} octets")
print(f"gain :   {taille_normal - taille_slotte} octets")

### `@dataclass(slots=True)`

Depuis Python 3.10, `@dataclass(slots=True)` génère `__slots__` automatiquement à partir des annotations. C'est la forme recommandée.

In [ ]:
from dataclasses import dataclass

@dataclass(slots=True, frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(p)
print(Point.__slots__)

---

## 3. `__slots__` et héritage

Règles :

1. Si une classe parente **n'a pas** `__slots__`, toutes ses sous-classes auront `__dict__`    (le gain est perdu).
2. Pour hériter proprement, chaque classe de la chaîne doit déclarer `__slots__`    (éventuellement `__slots__ = ()` si elle n'ajoute rien).
3. En héritage multiple, `__slots__` est contraignant : au plus **une** classe parente peut    ajouter de nouvelles cases slot (les autres doivent être `__slots__ = ()`).

In [ ]:
class A:
    __slots__ = ("x",)

class B(A):
    __slots__ = ("y",)

b = B()
b.x = 1
b.y = 2
print(b.x, b.y)

try:
    b.z = 3
except AttributeError as e:
    print("refusé :", e)

In [ ]:
# Si un parent oublie __slots__, le gain est perdu silencieusement
class AOups: pass  # pas de __slots__

class BSlot(AOups):
    __slots__ = ("x",)

b = BSlot()
b.x = 1
b.y = 2   # passe ! car AOups a donné un __dict__
print(b.__dict__)

---

## 4. `weakref` — références faibles

CPython libère un objet dès que son **compteur de références** tombe à zéro. Un cache qui garde une référence forte vers tous les objets empêche leur libération.

`weakref` crée une référence qui **ne maintient pas l'objet en vie** : dès que l'objet n'a plus de référence forte ailleurs, il est collecté, et la weakref devient invalide (`None` au déréférencement).

In [ ]:
import weakref

class Gros:
    def __init__(self, nom: str) -> None:
        self.nom = nom
    def __repr__(self) -> str:
        return f"Gros({self.nom!r})"

g = Gros("alice")
ref = weakref.ref(g)
print(ref())       # <Gros ...>
del g
print(ref())       # None : l'objet a été libéré

### Attention : `__slots__` + weakref

Pour qu'un objet avec `__slots__` puisse avoir des weakrefs, il faut inclure `"__weakref__"` dans `__slots__` (c'est automatique avec `@dataclass(slots=True, weakref_slot=True)`).

In [ ]:
class AvecWeakref:
    __slots__ = ("x", "__weakref__")
    def __init__(self, x: int) -> None: self.x = x

o = AvecWeakref(1)
import weakref
print(weakref.ref(o)())

---

## 5. `WeakValueDictionary` — caches qui ne retiennent pas

Un `WeakValueDictionary` est un dict dont les **valeurs** sont des references faibles. C'est le bon outil pour un cache **d'interning** : les objets disparaissent dès que plus personne ne les référence ailleurs.

In [ ]:
import weakref

class Token:
    def __init__(self, nom: str) -> None:
        self.nom = nom
    def __repr__(self) -> str:
        return f"Token({self.nom!r})"

cache: "weakref.WeakValueDictionary[str, Token]" = weakref.WeakValueDictionary()

def get_token(nom: str) -> Token:
    if nom not in cache:
        cache[nom] = Token(nom)
    return cache[nom]

t1 = get_token("a")
t2 = get_token("a")
print(t1 is t2)        # True : même objet
del t1, t2
import gc; gc.collect()
print(list(cache))     # [] : le cache s'est vidé tout seul

---

## 6. Le GC CPython

CPython combine **deux mécanismes** :

1. **Comptage de références.** Chaque objet a un `refcount` ; dès qu'il tombe à 0, l'objet est    libéré immédiatement et son `__del__` appelé. Rapide et déterministe.
2. **Cycle collector.** Le comptage seul ne peut pas libérer les **cycles** (A → B → A). Un    collector générationnel s'exécute périodiquement pour les trouver et les briser.


In [ ]:
import sys

a = [1, 2, 3]
print("refcount initial :", sys.getrefcount(a))
b = a
print("après b = a :", sys.getrefcount(a))
del b
print("après del b :", sys.getrefcount(a))

### Cycle qui ne peut pas être cassé par le refcount


In [ ]:
import gc

class Noeud:
    def __init__(self, nom: str) -> None:
        self.nom = nom
        self.ref = None
    def __del__(self) -> None:
        print(f"libéré : {self.nom}")

a = Noeud("a")
b = Noeud("b")
a.ref = b
b.ref = a   # cycle !
del a, b
print("(pas encore libérés : refcount = 1 chacun)")

gc.collect()
print("(après gc.collect : libérés)")

### Inspecter le GC

In [ ]:
import gc

print("seuils (génération 0, 1, 2) :", gc.get_threshold())
print("compteurs courants :", gc.get_count())
print("objets suivis par le GC :", len(gc.get_objects()))

---

## 7. Diagnostiquer une fuite mémoire

Une **fuite** classique en Python : un cache (ou un registry) qui continue de référencer des objets alors qu'ils ne sont plus utilisés. Le GC ne peut pas les libérer parce qu'il y a encore une **référence forte**.

Stratégie :

1. Mesurer l'évolution du nombre d'instances avec `len(gc.get_objects())` ou `tracemalloc` (cf. J4).
2. Identifier le type qui grossit anormalement.
3. Trouver qui retient les instances avec `gc.get_referrers(obj)`.
4. Corriger avec un `weakref`, un `WeakValueDictionary`, ou un vrai cycle de vie (close, __exit__).

In [ ]:
import gc

class Leaky:
    _registry = []
    def __init__(self, n: int) -> None:
        self.n = n
        Leaky._registry.append(self)   # fuite !

for i in range(5):
    Leaky(i)

n = sum(1 for o in gc.get_objects() if isinstance(o, Leaky))
print(f"instances vivantes : {n}")

# Nettoyage pour le reste du notebook
Leaky._registry.clear()

### La correction idiomatique : `WeakSet`

In [ ]:
import weakref

class Clean:
    _instances: "weakref.WeakSet[Clean]" = weakref.WeakSet()
    def __init__(self, n: int) -> None:
        self.n = n
        Clean._instances.add(self)

for i in range(5):
    Clean(i)

import gc; gc.collect()
print("instances vivantes :", len(Clean._instances))

---

## 8. Synthèse

| Outil | Pour quoi | À éviter quand |
|---|---|---|
| `__slots__` | économie mémoire sur beaucoup d'instances | classes avec attributs dynamiques |
| `@dataclass(slots=True)` | la forme moderne, recommandée | — |
| `weakref.ref(obj)` | référence qui ne retient pas | objets immutables (int, str, tuple) |
| `WeakValueDictionary` | cache / interning | quand on veut **garder** les valeurs |
| `gc.collect()` | libérer les cycles | en boucle chaude (coûteux) |
| `gc.get_referrers(obj)` | debug de fuite | en prod (lent) |


---

## 9. Exercices

### Exercice 1 — `Point` slotte *(facile)*

Écrire `Point(x: float, y: float)` avec `@dataclass(slots=True, frozen=True)`. Vérifier qu'on ne peut pas ajouter d'attribut.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Slots_weakref_gc", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass

@dataclass(slots=True, frozen=True)
class Point:
    x: float
    y: float

p = Point(1, 2)
print(p)
try:
    object.__setattr__(p, "z", 3)
except AttributeError as e:
    print("refusé :", e)
```

</details>


### Exercice 2 — Mesurer le gain de `__slots__` *(moyen)*

Créer deux classes identiques (`Normal` et `Slotte`) à 5 attributs. Créer 100 000 instances de chaque et comparer la consommation mémoire du processus avec `tracemalloc.get_traced_memory()`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Slots_weakref_gc", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import tracemalloc

class Normal:
    def __init__(self, a, b, c, d, e) -> None:
        self.a, self.b, self.c, self.d, self.e = a, b, c, d, e

class Slotte:
    __slots__ = ("a", "b", "c", "d", "e")
    def __init__(self, a, b, c, d, e) -> None:
        self.a, self.b, self.c, self.d, self.e = a, b, c, d, e

N = 100_000

tracemalloc.start()
xs = [Normal(1, 2, 3, 4, 5) for _ in range(N)]
cur_normal, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
ys = [Slotte(1, 2, 3, 4, 5) for _ in range(N)]
cur_slotte, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Normal : {cur_normal / 1024 / 1024:.2f} Mo")
print(f"Slotte : {cur_slotte / 1024 / 1024:.2f} Mo")
print(f"gain :   {(cur_normal - cur_slotte) / 1024 / 1024:.2f} Mo")
```

</details>


### Exercice 3 — `WeakValueDictionary` pour un interner *(difficile)*

Écrire une classe `Symbol` telle que `Symbol('foo') is Symbol('foo')` renvoie toujours `True`, mais sans garder les symboles en mémoire une fois qu'aucune autre référence n'existe. Utiliser `WeakValueDictionary`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Slots_weakref_gc", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import weakref

class Symbol:
    _cache: "weakref.WeakValueDictionary[str, Symbol]" = weakref.WeakValueDictionary()
    __slots__ = ("name", "__weakref__")

    def __new__(cls, name: str) -> "Symbol":
        existing = cls._cache.get(name)
        if existing is not None:
            return existing
        instance = object.__new__(cls)
        object.__setattr__(instance, "name", name)
        cls._cache[name] = instance
        return instance

    def __repr__(self) -> str:
        return f"Symbol({self.name!r})"

a = Symbol("foo")
b = Symbol("foo")
print(a is b)
```

</details>


### Exercice 4 — Chasse à la fuite *(deep dive)*

Le code ci-dessous laisse des `Ressource` en mémoire indéfiniment. Identifier qui les retient avec `gc.get_referrers`, puis corriger sans changer la signature des fonctions.

In [ ]:
# Code à déboguer
import gc

HANDLERS: list = []

class Ressource:
    def __init__(self, n: int) -> None: self.n = n

def on_event(r: Ressource) -> None:
    HANDLERS.append(lambda: r.n)   # closure sur r !

for i in range(3):
    on_event(Ressource(i))

vivantes = [o for o in gc.get_objects() if isinstance(o, Ressource)]
print("fuite :", len(vivantes))


<details>
<summary>📖 Voir la correction</summary>

```python
# La closure `lambda: r.n` retient `r` via la closure.
# Solution : ne pas capturer toute la ressource, juste la valeur utile.
import gc

HANDLERS: list = []

class Ressource:
    def __init__(self, n: int) -> None: self.n = n

def on_event(r: Ressource) -> None:
    n = r.n                       # extraction immédiate
    HANDLERS.append(lambda n=n: n)

for i in range(3):
    on_event(Ressource(i))

gc.collect()
vivantes = [o for o in gc.get_objects() if isinstance(o, Ressource)]
print("vivantes :", len(vivantes))
```

</details>


---

## Ressources

- [docs — `__slots__`](https://docs.python.org/3/reference/datamodel.html#slots)
- [docs — `weakref`](https://docs.python.org/3/library/weakref.html)
- [docs — `gc`](https://docs.python.org/3/library/gc.html)
- Instagram Engineering — *Dismissing Python Garbage Collection at Instagram*
